In [2]:
def conectar_base_datos(server='172.16.1.33', database='CUN_REPOSITORIO'):
    """
    Establece una conexión con SQL Server usando Trusted Connection.
    Retorna el objeto de conexión (conn) si es exitoso, o None si falla.
    """
    import pyodbc
    
    # Cadena de conexión con Driver de SQL Server estándar
    conn_str = f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'
    
    try:
        # Intentamos conectar con un tiempo límite de 10 segundos
        conn = pyodbc.connect(conn_str, timeout=10)
        print(f"✅ CONEXIÓN EXITOSA AL SERVIDOR: {server}")
        return conn

    except pyodbc.Error as e:
        print("❌ ERROR DE CONEXIÓN (pyodbc):")
        print(f"Detalle: {e}")
        return None

    except Exception as e:
        print(f"❌ OCURRIÓ UN ERROR INESPERADO: {e}")
        return None

In [7]:
from tabulate import tabulate

# 1. Conectar
conexion = conectar_base_datos()
cursor = conexion.cursor()

# 2. Consulta
query = "SELECT TOP 10 * FROM zoho.Base_Personas"
cursor.execute(query)

# 3. Traer resultados 👈 (esto te faltaba)
resultados = cursor.fetchall()

# 4. Obtener columnas
columnas = [col[0] for col in cursor.description]

# 5. Mostrar
print(tabulate(resultados, headers=columnas, tablefmt="grid"))

# 6. Cerrar
cursor.close()
conexion.close()

✅ CONEXIÓN EXITOSA AL SERVIDOR: 172.16.1.33
+-----------+--------+----------------+---------------+------------------------+--------------------+----------------------+------------+----------------------------------------------------+---------------+--------------+----------------+----------------+----------------------------------------------------+----------------+-------------+------------+------------+---------------+-----------------------------------+------------------------------------+-----------------------------+------------------------------------+--------------+---------------+------------+---------------+------------+-----------------+------------+-------------+-----------+-------------------------------------------------------------+----------------+---------------------------------------------+----------+-------------+---------------------+------------+----------+--------------+------------------+---------+-------------+----------------+--------------------+-------------

In [9]:
from tabulate import tabulate

# 1. Conectar
conexion = conectar_base_datos()
cursor = conexion.cursor()

# 2. Query
query = """
WITH datos AS (
    SELECT 
        CAST(DOC_ALUM AS VARCHAR(50)) AS DOC_ALUM,
        COALESCE(EMAIL_INS, EMAIL, EMAIL_PER) AS EMAIL_INS,
        NOM_PROGRAMA,
        MODALIDAD,
        CIUDAD,
        DEPARTAMENTO,
        PERIODO,
        ROW_NUMBER() OVER (
            PARTITION BY DOC_ALUM
            ORDER BY 
                CASE 
                    WHEN MODALIDAD = 'Presencial' THEN 1
                    WHEN MODALIDAD = 'Virtual' THEN 2
                    ELSE 3
                END,
                PERIODO DESC
        ) AS rn
    FROM zoho.Base_Personas
    WHERE CAST(DOC_ALUM AS VARCHAR(50)) IN (
        '1028863261','1011086220','1031171124','1010156798','1233501755',
        '1016716433','1013108384','1027151205','1025530709','1000850787',
        '1072961025','1024504661','1011089116','1101685660','1082979523',
        '1030664643','10967574','1021315566','1003721769','1042433398',
        '1072425446','1082958380','1001186234','1070008398'
    )
    AND NOM_PROGRAMA NOT LIKE '%EDUCACION CONTINUADA%'
)

SELECT 
    DOC_ALUM,
    EMAIL_INS,
    NOM_PROGRAMA,
    MODALIDAD,
    CIUDAD,
    DEPARTAMENTO
FROM datos
WHERE rn = 1
"""

# 3. Ejecutar
cursor.execute(query)

# 4. Resultados
resultados = cursor.fetchall()
columnas = [col[0] for col in cursor.description]

# 5. Mostrar
print(tabulate(resultados, headers=columnas, tablefmt="grid"))

# 6. Cerrar
cursor.close()
conexion.close()

✅ CONEXIÓN EXITOSA AL SERVIDOR: 172.16.1.33
+------------+-------------------------------+------------------------------------------------+-------------+-------------+------------------+
|   DOC_ALUM | EMAIL_INS                     | NOM_PROGRAMA                                   | MODALIDAD   | CIUDAD      | DEPARTAMENTO     |
+============+===============================+================================================+=============+=============+==================+
| 1000850787 | juan.velandiag@cun.edu.co     | INGENIERIA DE SISTEMAS                         | Presencial  | BOGOTA      | DISTRITO CAPITAL |
+------------+-------------------------------+------------------------------------------------+-------------+-------------+------------------+
| 1001186234 | carlos.povedan@cun.edu.co     | INGENIERIA DE SISTEMAS                         | Presencial  | BOGOTA      | DISTRITO CAPITAL |
+------------+-------------------------------+------------------------------------------------+---